In [ ]:
SOURCE = 'import base64, hashlib, json, sys, zlib\nfrom pathlib import Path\nPORTABLE_SOURCE=\'"""Preserve daily errors for exact, disjoint Development collection reconciliation.\\n\\nExports reverify original artifacts, never infer paired errors from summary metrics.\\nMounted receiver paths are relocated explicitly without changing any content hash.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport copy\\nimport json\\nimport math\\nfrom pathlib import Path, PurePosixPath\\n\\nfrom tools import hung_results as scorer\\n\\nSCORER_SHA = "68fccbed056734d9b4842f7bda088880d7812d12417d64878d1aa018fc010456"\\nPLAN_SHA = "2d448d2584f65baf7e09445e013b4283314b08ea2cce855c6850e9fc12ba87ee"\\nMODEL_CODE = "44b8532959d76fb20691881539f991ca44983727"\\n\\n\\ndef require(condition: bool, message: str) -> None:\\n    if not condition:\\n        raise ValueError(message)\\n\\n\\ndef load_pin(pin: dict, root: Path) -> dict:\\n    return json.loads(scorer.verified_file(pin, root).read_bytes())\\n\\n\\ndef export_catalog(catalog: dict, root: Path, provenance: dict) -> dict:\\n    """Recheck manifests/checkpoints/proofs and retain per-day original-unit errors."""\\n    require(scorer.file_sha(Path(scorer.__file__)) == SCORER_SHA, "Scorer source changed")\\n    plan_pin = catalog["spec"]["plan"]\\n    plan = load_pin(plan_pin, root)\\n    require(plan["scope"] == "development_only", "Foreign plan scope")\\n    if "receiver" in provenance:\\n        require(plan_pin["sha256"] == provenance["receiver"]["plan_sha256"], "Runtime plan differs")\\n    else:\\n        require(plan_pin["sha256"] == PLAN_SHA, "Canonical plan differs")\\n    records = scorer.verify_artifacts(catalog, root)\\n    return {\\n        "schema": "hung-portable-daily-errors-v1",\\n        "scope": "development_only",\\n        "canonical_plan_sha256": PLAN_SHA,\\n        "scorer_sha256": SCORER_SHA,\\n        "provenance": provenance,\\n        "acceptance": "Technical export only; independent review and full coverage pending",\\n        "records": records,\\n    }\\n\\n\\ndef relocate_catalog(catalog: dict, original: str, mounted: Path) -> dict:\\n    """Map exact original receiver paths to a read-only mount, preserving SHA pins."""\\n    prefix = PurePosixPath(original)\\n    require(prefix.is_absolute() and ".." not in prefix.parts, "Invalid original root")\\n    relocated = copy.deepcopy(catalog)\\n\\n    def visit(value):\\n        if isinstance(value, dict):\\n            for key, item in value.items():\\n                if key == "path":\\n                    path = PurePosixPath(item)\\n                    require(".." not in path.parts and path.is_relative_to(prefix), "Foreign path")\\n                    value[key] = str(mounted / path.relative_to(prefix))\\n                else:\\n                    visit(item)\\n        elif isinstance(value, list):\\n            for item in value:\\n                visit(item)\\n\\n    visit(relocated)\\n    return relocated\\n\\n\\ndef export_receiver(mounted: Path, proof_sha256: str) -> dict:\\n    """Reverify an already completed CPU receiver from its pinned mounted output."""\\n    proof = load_pin(\\n        {"path": str(mounted / "receiver-proof.json"), "sha256": proof_sha256}, mounted\\n    )\\n    require(\\n        proof["scope"] == "development_only" and proof["passed"] and proof["executed"] is False,\\n        "Receiver did not pass without training",\\n    )\\n    require(proof["model_code_commit"] == MODEL_CODE, "Model provenance differs")\\n    require(proof["scorer_sha256"] == SCORER_SHA, "Receiver scorer differs")\\n    require(proof["derivation"]["parent_plan_sha256"] == PLAN_SHA, "Canonical plan differs")\\n    terminal = proof["verification"]\\n    require(terminal["exit_code"] == 0 and terminal["executed"] is False, "Verifier failed")\\n    command = terminal["command"]\\n    require(\\n        "--verify-only" in command and "--execute" not in command, "Execution command forbidden"\\n    )\\n    evidence = mounted / "cpu-readback"\\n    require(scorer.file_sha(evidence / "verify.log") == terminal["log_sha256"], "Log differs")\\n    require(json.loads((evidence / "verify-exit.json").read_bytes()) == terminal, "Exit differs")\\n    catalog = load_pin(\\n        {"path": str(evidence / "catalog.json"), "sha256": proof["catalog_sha256"]}, mounted\\n    )\\n    original = str(PurePosixPath(command[command.index("--plan") + 1]).parents[3])\\n    # --plan points at <root>/repo/.cache/provider/plan.json.\\n    require(PurePosixPath(original).name == mounted.name, "Explicit mount root identity differs")\\n    catalog = relocate_catalog(catalog, original, mounted)\\n    require(len(catalog["runs"]) == proof["jobs"], "Receiver catalog count differs")\\n    return export_catalog(\\n        catalog,\\n        mounted,\\n        {"receiver_proof_sha256": proof_sha256, "receiver": proof, "original_root": original},\\n    )\\n\\n\\ndef merge_collections(spec: dict, root: Path) -> dict:\\n    """Merge exact expected job sets; duplicate anchors and altered errors fail closed."""\\n    require(spec.get("scope") == "development_only", "Development-only specification required")\\n    records, ids, logical = [], set(), set()\\n    for collection in spec["collections"]:\\n        value = load_pin(collection["export"], root)\\n        require(value["schema"] == "hung-portable-daily-errors-v1", "Unsupported export")\\n        require(value["scope"] == "development_only", "Foreign scope")\\n        require(value["canonical_plan_sha256"] == PLAN_SHA, "Plan differs")\\n        require(value["scorer_sha256"] == SCORER_SHA, "Scorer differs")\\n        rows = value["records"]\\n        expected = collection["job_ids"]\\n        require(len(expected) == len(set(expected)) and bool(expected), "Invalid expected job set")\\n        require(\\n            len(rows) == len(expected) and {r["job_id"] for r in rows} == set(expected),\\n            "Jobs differ",\\n        )\\n        for row in rows:\\n            identity = (row["method"], row["budget"], row["seed"], row["entity_id"])\\n            require(row["job_id"] not in ids and identity not in logical, "Duplicate logical run")\\n            require(row["status"] in {"DISPATCH_VERIFIED", "VERIFIED_REUSE"}, "Unmatched run")\\n            require(row["coverage"]["scored"] == len(row["days"]), "Scored coverage differs")\\n            for day in row["days"].values():\\n                require(\\n                    all(math.isfinite(day[k]) and day[k] >= 0 for k in ("absolute", "squared")),\\n                    "Invalid daily errors",\\n                )\\n                require(len(day["label_sha256"]) == 64, "Label hash missing")\\n            calculated = scorer.metrics(list(row["days"].values()))\\n            require((calculated is None) == (row["metrics"] is None), "Metric availability differs")\\n            if calculated is not None:\\n                require(\\n                    all(\\n                        math.isclose(v, row["metrics"][k], rel_tol=1e-9, abs_tol=1e-9)\\n                        for k, v in calculated.items()\\n                    ),\\n                    "Daily errors and metrics differ",\\n                )\\n            ids.add(row["job_id"])\\n            logical.add(identity)\\n            records.append(row)\\n    require(bool(records), "No verified records")\\n    pairs, blocked = scorer.paired_results(records)\\n    return {\\n        "schema": "hung-development-partial-results-v2",\\n        "scope": "development_only",\\n        "acceptance": "Provisional; full coverage and independent review required",\\n        "catalog_sha256": spec["spec_sha256"],\\n        "collection_proofs": spec["collections"],\\n        "runs": [{k: v for k, v in r.items() if k != "days"} for r in records],\\n        "counts": {\\n            "runs": len(records),\\n            "scorable_runs": sum(bool(r["days"]) for r in records),\\n            "zero_scored_runs": sum(not r["days"] for r in records),\\n        },\\n        "paired_cells": pairs,\\n        "blocked_pairs": blocked,\\n        "station_first": scorer.station_first(pairs),\\n        "aggregation": "Exact daily label-hash intersection within station/seed; equal seed means within station, then equal station means within each budget. Disjoint expected job sets; no global ranking or Final selection.",\\n    }\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("action", choices=("export", "receiver", "merge"))\\n    parser.add_argument("--input", type=Path, required=True)\\n    parser.add_argument("--sha256", required=True)\\n    parser.add_argument("--root", type=Path, required=True)\\n    parser.add_argument("--output", type=Path, required=True)\\n    args = parser.parse_args()\\n    require(args.output.resolve().is_relative_to(args.root.resolve()), "Output outside project")\\n    if args.action == "receiver":\\n        require(args.input.resolve().is_relative_to(args.root.resolve()), "Mount outside project")\\n        result = export_receiver(args.input, args.sha256)\\n    else:\\n        value = load_pin({"path": str(args.input), "sha256": args.sha256}, args.root)\\n        if args.action == "export":\\n            result = export_catalog(value, args.root, {"catalog_sha256": args.sha256})\\n        else:\\n            result = merge_collections({**value, "spec_sha256": args.sha256}, args.root)\\n    with args.output.open("x", encoding="utf-8", newline="\\\\n") as stream:\\n        json.dump(result, stream, sort_keys=True, indent=2, allow_nan=False)\\n        stream.write("\\\\n")\\n    print(json.dumps({"output": str(args.output), "sha256": scorer.file_sha(args.output)}))\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\'\nSCORER_SOURCE=\'"""Read-only Development forecast scoring from SHA-pinned, verified artifacts.\\n\\nRecompute original-unit errors from stored issued forecasts and already arrived\\nlabels. Never load Final inputs or choose configurations. All tables are partial\\nuntil the separate coverage, fairness and independent-review gates pass.\\n"""\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport math\\nimport struct\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nimport torch\\n\\n\\ndef file_sha(path: Path) -> str:\\n    with path.open("rb") as stream:\\n        return hashlib.file_digest(stream, "sha256").hexdigest()\\n\\n\\ndef verified_file(pin: dict, root: Path) -> Path:\\n    path = Path(pin["path"]).resolve()\\n    if not path.is_relative_to(root.resolve()) or file_sha(path) != pin["sha256"]:\\n        raise ValueError("Artifact outside project or SHA mismatch")\\n    return path\\n\\n\\ndef pin(path: Path, root: Path) -> dict:\\n    path = path.resolve()\\n    if not path.is_relative_to(root.resolve()):\\n        raise ValueError("Artifact outside project")\\n    return {"path": str(path), "sha256": file_sha(path)}\\n\\n\\ndef method_family(job: dict) -> str:\\n    suffix = "_" + job["cell"]\\n    if not job["id"].endswith(suffix) or len(job["id"]) <= len(suffix):\\n        raise ValueError("Job family/cell naming differs")\\n    return job["id"][: -len(suffix)]\\n\\n\\ndef build_catalog(spec: dict, root: Path) -> dict:\\n    """Bind complete dispatcher receipts and VERIFIED_REUSE rows to read-only metrics."""\\n    plan = json.loads(verified_file(spec["plan"], root).read_bytes())\\n    partition = json.loads(verified_file(spec["partition"], root).read_bytes())\\n    if plan["scope"] != "development_only" or partition["plan_sha256"] != spec["plan"]["sha256"]:\\n        raise ValueError("Plan/partition binding differs")\\n    cells = {c["id"]: c for c in plan["cells"]}\\n    jobs = {j["id"]: j for j in plan["jobs"]}\\n    configs = {c["id"]: c for c in plan["configs"]}\\n    output = Path(spec.get("readback_output_root", plan["output_root"])).resolve()\\n    rows = []\\n\\n    def entry(job_id, method, cell, manifest_pin, checkpoint_pin, source, target, proofs, status):\\n        return {\\n            "job_id": job_id,\\n            "method": method,\\n            "budget": cell["budget"],\\n            "seed": cell["seed"],\\n            "entity_id": cell["entity_id"],\\n            "input_manifest_sha256": cell["input_manifest_sha256"],\\n            "source_sha256": source["sha256"] if source else None,\\n            "target_sha256": target["sha256"] if target else None,\\n            "manifest": manifest_pin,\\n            "checkpoint": checkpoint_pin,\\n            "proofs": proofs,\\n            "status": status,\\n        }\\n\\n    if spec.get("reuse_ledger"):\\n        ledger = json.loads(verified_file(spec["reuse_ledger"], root).read_bytes())\\n        for row in ledger["jobs"]:\\n            if row["status"] != "VERIFIED_REUSE":\\n                continue\\n            if not row["job_id"].startswith("main_"):\\n                raise ValueError("Unsupported reuse mapping")\\n            cell = cells[row["job_id"][len("main_") :]]\\n            manifest_path = verified_file(row["run_manifest"], root)\\n            manifest = json.loads(manifest_path.read_bytes())\\n            checkpoint = {\\n                "path": str(manifest_path.with_name("runner_checkpoint.pt")),\\n                "sha256": manifest["final_runner_checkpoint_sha256"],\\n            }\\n            proofs = [spec["reuse_ledger"], spec["plan"], row["source"], row["target"]]\\n            journal = manifest["event_journal"]\\n            proofs.append(\\n                {"path": str(manifest_path.with_name(journal["file"])), "sha256": journal["sha256"]}\\n            )\\n            rows.append(\\n                entry(\\n                    row["job_id"],\\n                    "main",\\n                    cell,\\n                    row["run_manifest"],\\n                    checkpoint,\\n                    row["source"],\\n                    row["target"],\\n                    proofs,\\n                    "VERIFIED_REUSE",\\n                )\\n            )\\n    for completed in spec["completed_shards"]:\\n        exit_record = json.loads(verified_file(completed["verify_exit"], root).read_bytes())\\n        if exit_record["exit_code"] != 0:\\n            raise ValueError("Shard verifier did not succeed")\\n        shard = next(s for s in partition["shards"] if s["id"] == completed["id"])\\n        for job_id in shard["job_ids"]:\\n            job, cell = jobs[job_id], cells[jobs[job_id]["cell"]]\\n            receipt_path = output / "receipts" / job_id / "job.json"\\n            receipt = json.loads(receipt_path.read_bytes())\\n            if (receipt["job_id"], receipt["plan_sha256"], receipt["shards_sha256"]) != (\\n                job_id,\\n                spec["plan"]["sha256"],\\n                spec["partition"]["sha256"],\\n            ):\\n                raise ValueError("Job receipt identity differs")\\n            config_pin = {k: configs[job["config"]][k] for k in ("path", "sha256")}\\n            config = json.loads(verified_file(config_pin, root).read_bytes())\\n            final_stage = f"replay-{config[\\\'replay\\\'][\\\'max_origins\\\']}"\\n            proofs = [\\n                spec["plan"],\\n                spec["partition"],\\n                completed["verify_exit"],\\n                pin(receipt_path, root),\\n                config_pin,\\n            ]\\n            for stage, sha in receipt["stages"].items():\\n                stage_pin = {"path": str(receipt_path.with_name(stage + ".json")), "sha256": sha}\\n                stage_record = json.loads(verified_file(stage_pin, root).read_bytes())\\n                binding = stage_record["binding"]\\n                if (\\n                    binding["job"],\\n                    binding["plan_sha256"],\\n                    binding["shards_sha256"],\\n                    binding["config_sha256"],\\n                    binding["input_manifest_sha256"],\\n                ) != (\\n                    job,\\n                    spec["plan"]["sha256"],\\n                    spec["partition"]["sha256"],\\n                    config_pin["sha256"],\\n                    cell["input_manifest_sha256"],\\n                ):\\n                    raise ValueError("Stage provenance differs")\\n                proofs.append(stage_pin)\\n                for name, record in stage_record["files"].items():\\n                    artifact = {\\n                        "path": str(output / "jobs" / job_id / stage / name),\\n                        "sha256": record["sha256"],\\n                    }\\n                    verified_file(artifact, root)\\n                    proofs.append(artifact)\\n            if final_stage not in receipt["stages"]:\\n                raise ValueError("Final replay stage incomplete")\\n            manifest_path = output / "jobs" / job_id / final_stage / "run_manifest.json"\\n            manifest = json.loads(manifest_path.read_bytes())\\n            checkpoint_name = (\\n                "runner.pt"\\n                if manifest["schema"] == "hung-single-baseline-replay-v1"\\n                else "runner_checkpoint.pt"\\n            )\\n            # The final checkpoint hash is already bound by the stage inventory.\\n            checkpoint = {\\n                "path": str(manifest_path.with_name(checkpoint_name)),\\n                "sha256": manifest["final_runner_checkpoint_sha256"],\\n            }\\n            for role in ("source", "target"):\\n                if job[role]:\\n                    proofs.append(job[role])\\n            rows.append(\\n                entry(\\n                    job_id,\\n                    method_family(job),\\n                    cell,\\n                    pin(manifest_path, root),\\n                    checkpoint,\\n                    job["source"],\\n                    job["target"],\\n                    proofs,\\n                    "DISPATCH_VERIFIED",\\n                )\\n            )\\n    return {\\n        "scope": "development_only",\\n        "schema": "hung-results-catalog-v1",\\n        "spec": spec,\\n        "runs": rows,\\n    }\\n\\n\\ndef metrics(days: list[dict]) -> dict | None:\\n    if not days:\\n        return None\\n    hours = 24 * len(days)\\n    return {\\n        "hours": hours,\\n        "mae_kwh": sum(d["absolute"] for d in days) / hours,\\n        "rmse_kwh": math.sqrt(sum(d["squared"] for d in days) / hours),\\n    }\\n\\n\\ndef score_saved(state: dict, manifest: dict) -> dict:\\n    """Score live predictions only; do not use candidate or audit performance."""\\n    if state["schema"] not in {"hung-sprint2-runner-v1", "hung-single-baseline-runner-v1"}:\\n        raise ValueError("Unsupported checkpoint schema")\\n    if manifest.get("status") != "completed":\\n        raise ValueError("Run not terminal")\\n    scale = state["energy_scale"]\\n    if not math.isfinite(scale) or scale <= 0:\\n        raise ValueError("Invalid original-unit scale")\\n    days = {}\\n    issued_ids = set()\\n    for issued in state["issued"]:\\n        day_id = issued["day_id"]\\n        if day_id in issued_ids:\\n            raise ValueError("Duplicate issued day")\\n        issued_ids.add(day_id)\\n        labels = issued.get("y_scaled")\\n        if labels is None:\\n            if issued.get("normalized_mae") is not None:\\n                raise ValueError("Missing labels carry a score")\\n            continue\\n        if issued.get("normalized_mae") is None:\\n            raise ValueError("Arrived labels were not scored")\\n        if state["schema"] == "hung-single-baseline-runner-v1":\\n            prediction = issued["prediction"]\\n        else:\\n            levels = state["controller"]["levels"]\\n            prediction = issued["forecasts"][levels.index(issued["active_level"])]\\n        prediction, labels = prediction.reshape(-1).double(), labels.reshape(-1).double()\\n        if prediction.shape != (24,) or labels.shape != (24,):\\n            raise ValueError("Incomplete 24-hour horizon")\\n        if not bool(torch.isfinite(prediction).all() and torch.isfinite(labels).all()):\\n            raise ValueError("Nonfinite forecast/label")\\n        if bool((prediction < 0).any() or (labels < 0).any()):\\n            raise ValueError("Negative forecast/label")\\n        error = (prediction - labels) * scale\\n        days[day_id] = {\\n            "absolute": float(error.abs().sum()),\\n            "squared": float(error.square().sum()),\\n            "label_sha256": hashlib.sha256(\\n                struct.pack("<24d", *(labels * scale).tolist())\\n            ).hexdigest(),\\n        }\\n    coverage = manifest["coverage"]\\n    if len(issued_ids) != coverage["created"] or len(days) != coverage["scored"]:\\n        raise ValueError("Checkpoint coverage differs")\\n    calculated = metrics(list(days.values()))\\n    declared = manifest["metrics_kwh"]\\n    if isinstance(declared, dict) and "live" in declared:\\n        declared = declared["live"]\\n    if (calculated is None) != (declared is None):\\n        raise ValueError("Declared metric availability differs")\\n    if calculated is not None:\\n        for key, value in calculated.items():\\n            if not math.isclose(value, declared[key], rel_tol=1e-9, abs_tol=1e-9):\\n                raise ValueError(f"Recomputed {key} differs")\\n    return {"coverage": coverage, "metrics": calculated, "days": days}\\n\\n\\ndef verify_artifacts(catalog: dict, root: Path) -> list[dict]:\\n    if catalog.get("scope") != "development_only":\\n        raise ValueError("Development-only catalog required")\\n    records, identities = [], set()\\n    for entry in catalog["runs"]:\\n        if entry["status"] not in {"VERIFIED_REUSE", "DISPATCH_VERIFIED"}:\\n            raise ValueError("Unmatched or unverified run cannot enter metric comparison")\\n        identity = (entry["method"], entry["budget"], entry["seed"], entry["entity_id"])\\n        if identity in identities:\\n            raise ValueError("Duplicate logical run")\\n        identities.add(identity)\\n        for proof in entry["proofs"]:\\n            verified_file(proof, root)\\n        manifest = json.loads(verified_file(entry["manifest"], root).read_bytes())\\n        checkpoint = verified_file(entry["checkpoint"], root)\\n        if manifest["final_runner_checkpoint_sha256"] != entry["checkpoint"]["sha256"]:\\n            raise ValueError("Manifest/checkpoint binding differs")\\n        binding = manifest["input_binding"]\\n        if binding["input_manifest_sha256"] != entry["input_manifest_sha256"]:\\n            raise ValueError("Input binding differs")\\n        if binding["entity_id"] != entry["entity_id"] or binding["history_days"] != entry["budget"]:\\n            raise ValueError("Station/budget binding differs")\\n        seed = manifest["config"].get("seed")\\n        baseline_receipt_seed = (\\n            seed is None\\n            and entry["status"] == "DISPATCH_VERIFIED"\\n            and entry["method"] in {"seasonal_naive", "frozen_source", "target_only"}\\n        )\\n        if (seed != entry["seed"] and not baseline_receipt_seed) or not entry["proofs"]:\\n            raise ValueError("Seed or verification proof missing")\\n        for role in ("source", "target"):\\n            expected = entry.get(role + "_sha256")\\n            if expected is not None and manifest.get(role + "_checkpoint_sha256") != expected:\\n                raise ValueError("Source/Target binding differs")\\n        state = torch.load(checkpoint, map_location="cpu", weights_only=True)\\n        if state["binding"] != binding:\\n            raise ValueError("Checkpoint input binding differs")\\n        scored = score_saved(state, manifest)\\n        records.append(\\n            {\\n                **entry,\\n                **scored,\\n                "code_commit": manifest["code"]["commit"],\\n                "device": manifest.get("device"),\\n                "final_stage_duration_seconds": manifest.get("duration_seconds"),\\n                "cumulative_runtime_costs": manifest.get("runtime_costs", manifest.get("cost")),\\n                "cumulative_proxy_costs": manifest.get("proxy_costs"),\\n            }\\n        )\\n    return records\\n\\n\\ndef paired_results(records: list[dict]) -> tuple[list[dict], list[dict]]:\\n    cells = defaultdict(dict)\\n    for record in records:\\n        cells[(record["budget"], record["seed"], record["entity_id"])][record["method"]] = record\\n    pairs, blocked = [], []\\n    for (budget, seed, station), methods in sorted(cells.items()):\\n        for reference in ("main", "target_only"):\\n            if reference not in methods:\\n                continue\\n            baseline = methods[reference]\\n            for method, candidate in sorted(methods.items()):\\n                if method == reference:\\n                    continue\\n                reason = None\\n                if candidate["input_manifest_sha256"] != baseline["input_manifest_sha256"]:\\n                    reason = "input_mismatch"\\n                for role in ("source_sha256", "target_sha256"):\\n                    left, right = candidate.get(role), baseline.get(role)\\n                    if left is not None and right is not None and left != right:\\n                        reason = "initialization_mismatch"\\n                common = sorted(candidate["days"].keys() & baseline["days"].keys())\\n                if any(\\n                    candidate["days"][d]["label_sha256"] != baseline["days"][d]["label_sha256"]\\n                    for d in common\\n                ):\\n                    reason = "label_mismatch"\\n                if not common:\\n                    reason = reason or "no_paired_scored_days"\\n                identity = {\\n                    "budget": budget,\\n                    "seed": seed,\\n                    "entity_id": station,\\n                    "method": method,\\n                    "reference": reference,\\n                }\\n                if reason:\\n                    blocked.append({**identity, "reason": reason})\\n                    continue\\n                left = metrics([candidate["days"][d] for d in common])\\n                right = metrics([baseline["days"][d] for d in common])\\n                pairs.append(\\n                    {\\n                        **identity,\\n                        "paired_days": len(common),\\n                        "method_metrics": left,\\n                        "reference_metrics": right,\\n                        "uses_source": candidate.get("source_sha256") is not None,\\n                        "paired_day_ids_sha256": hashlib.sha256(\\n                            json.dumps(common).encode()\\n                        ).hexdigest(),\\n                    }\\n                )\\n    return pairs, blocked\\n\\n\\ndef station_first(pairs: list[dict]) -> list[dict]:\\n    stations = defaultdict(list)\\n    for pair in pairs:\\n        key = (pair["budget"], pair["method"], pair["reference"], pair["entity_id"])\\n        stations[key].append(pair)\\n    groups = defaultdict(list)\\n    for (budget, method, reference, station), rows in sorted(stations.items()):\\n        means = {\\n            side: {\\n                metric: sum(r[side][metric] for r in rows) / len(rows)\\n                for metric in ("mae_kwh", "rmse_kwh")\\n            }\\n            for side in ("method_metrics", "reference_metrics")\\n        }\\n        groups[(budget, method, reference)].append(\\n            {\\n                "entity_id": station,\\n                "seeds": len(rows),\\n                "uses_source": all(r["uses_source"] for r in rows),\\n                **means,\\n            }\\n        )\\n    result = []\\n    for (budget, method, reference), rows in sorted(groups.items()):\\n        averages = {\\n            side: {\\n                metric: sum(r[side][metric] for r in rows) / len(rows)\\n                for metric in ("mae_kwh", "rmse_kwh")\\n            }\\n            for side in ("method_metrics", "reference_metrics")\\n        }\\n        gain = averages["reference_metrics"]["mae_kwh"] - averages["method_metrics"]["mae_kwh"]\\n        denominator = averages["reference_metrics"]["mae_kwh"]\\n        worse = sum(\\n            r["method_metrics"]["mae_kwh"] > r["reference_metrics"]["mae_kwh"] for r in rows\\n        ) / len(rows)\\n        is_transfer = reference == "target_only" and all(r["uses_source"] for r in rows)\\n        result.append(\\n            {\\n                "budget": budget,\\n                "method": method,\\n                "reference": reference,\\n                "stations": len(rows),\\n                "minimum_seeds_per_station": min(r["seeds"] for r in rows),\\n                **averages,\\n                "absolute_mae_gain_kwh": gain,\\n                "relative_mae_gain": gain / denominator if denominator > 0 else None,\\n                "worse_than_reference_station_fraction": worse,\\n                "negative_transfer_station_fraction": worse if is_transfer else None,\\n                "transfer_gain": gain / denominator if is_transfer and denominator > 0 else None,\\n            }\\n        )\\n    return result\\n\\n\\ndef aggregate_results(catalog: dict, root: Path) -> dict:\\n    records = verify_artifacts(catalog, root)\\n    pairs, blocked = paired_results(records)\\n    return {\\n        "schema": "hung-development-partial-results-v2",\\n        "scope": "development_only",\\n        "acceptance": "Provisional; full coverage and independent review required",\\n        "runs": [{k: v for k, v in r.items() if k != "days"} for r in records],\\n        "counts": {\\n            "runs": len(records),\\n            "scorable_runs": sum(bool(r["days"]) for r in records),\\n            "zero_scored_runs": sum(not r["days"] for r in records),\\n        },\\n        "paired_cells": pairs,\\n        "blocked_pairs": blocked,\\n        "station_first": station_first(pairs),\\n        "aggregation": "Complete hourly errors within paired station/seed; mean seeds within station, then equal station mean within each budget. Budgets remain separate. Transfer gain and negative transfer apply only to Source-using methods versus Target-only. Other comparisons report generic MAE differences/worse-than-reference fractions. Zero reference MAE gives undefined relative gain; absolute differences remain available.",\\n    }\\n\\n\\ndef main() -> None:\\n    from experiments.hung_sprint2.dispatch_state import atomic_create\\n\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument("--catalog", type=Path, required=True)\\n    parser.add_argument("--catalog-sha256", required=True)\\n    parser.add_argument("--project-root", type=Path, required=True)\\n    parser.add_argument("--output", type=Path, required=True)\\n    parser.add_argument(\\n        "--build-catalog", action="store_true", help="Input is a pinned collection specification"\\n    )\\n    args = parser.parse_args()\\n    path = verified_file(\\n        {"path": str(args.catalog), "sha256": args.catalog_sha256}, args.project_root\\n    )\\n    value = json.loads(path.read_bytes())\\n    result = (\\n        build_catalog(value, args.project_root)\\n        if args.build_catalog\\n        else aggregate_results(value, args.project_root)\\n    )\\n    result["catalog_sha256"] = args.catalog_sha256\\n    if not args.output.resolve().is_relative_to(args.project_root.resolve()):\\n        raise ValueError("Output outside project")\\n    atomic_create(args.output, result)\\n    print(\\n        json.dumps(\\n            {\\n                "output": str(args.output),\\n                "sha256": file_sha(args.output),\\n                **result.get("counts", {}),\\n            }\\n        )\\n    )\\n\\n\\nif __name__ == "__main__":\\n    main()\\n\'\nPROOF_SHA=\'16d39bce5ed7b5998204ca592490152d3810c550b658dab8dd76a2dcade6142b\'\nassert hashlib.sha256(PORTABLE_SOURCE.encode()).hexdigest()==\'f4cebdca4c6ca1319ae01ba0d0166cf4e71eee29c5d6c872f2be4b45f77726ec\'\nassert hashlib.sha256(SCORER_SOURCE.encode()).hexdigest()==\'68fccbed056734d9b4842f7bda088880d7812d12417d64878d1aa018fc010456\'\nimport torch\nassert not torch.cuda.is_available(), \'CPU export only\'\ncode=Path(\'/kaggle/working/portable-export-code-v1\')\n(code/\'tools\').mkdir(parents=True,exist_ok=False)\nfor name,content in [(\'__init__.py\',\'\'),(\'hung_results.py\',SCORER_SOURCE),(\'hung_portable_results.py\',PORTABLE_SOURCE)]:\n    with (code/\'tools\'/name).open(\'x\',encoding=\'utf-8\',newline=\'\\n\') as stream: stream.write(content)\nsys.path.insert(0,str(code))\nfrom tools import hung_portable_results as portable\nproofs=[p for p in Path(\'/kaggle/input\').rglob(\'receiver-proof.json\') if hashlib.sha256(p.read_bytes()).hexdigest()==PROOF_SHA]\nassert len(proofs)==1, \'Exact completed CPU receiver missing/ambiguous\'\nvalue=portable.export_receiver(proofs[0].parent,PROOF_SHA)\nbody=(json.dumps(value,sort_keys=True,separators=(\',\',\':\'),allow_nan=False)+\'\\n\').encode()\nwith Path(\'/kaggle/working/portable-shard0008-v1.json\').open(\'xb\') as stream: stream.write(body)\nencoded=base64.b64encode(zlib.compress(body)).decode(\'ascii\')\nchunks=[encoded[i:i+3000] for i in range(0,len(encoded),3000)]\nprint(json.dumps({\'portable_export\':{\'scope\':\'development_only\',\'executed\':False,\'jobs\':len(value[\'records\']),\'sha256\':hashlib.sha256(body).hexdigest(),\'bytes\':len(body),\'chunks\':len(chunks),\'receiver_proof_sha256\':PROOF_SHA}}),flush=True)\nfor i,data in enumerate(chunks): print(json.dumps({\'portable_chunk\':i,\'data\':data}),flush=True)\n'
exec(compile(SOURCE, "pinned_portable_export", "exec"))